# PrimeNet — aplasia finetune from mimic_all SSL (Colab)

Finetune on **`mimic_cohort_aplasia_45_days`** using the HPC **mimic_all** pretrain checkpoint.

**Upload to `/content/uploads/`:**
1. `aplasia_saved_data.zip` (from Desktop)
2. `checkpoint_best.bin` + `primenet_saved_variables.pkl` (FAU box `primenet_checkpoint`)

**Runtime:** GPU. No Google Drive required.

## 1. Clone + install

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "AhmedSofan10/ChemoTreeVsDL"
BRANCH = "primenet"
REPO_DIR = "/content/ChemoTreeVsDL"

if not Path(REPO_DIR).is_dir():
    subprocess.check_call(["git", "clone", "-b", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", BRANCH])

os.chdir(REPO_DIR)
sys.path.insert(0, str(Path.cwd()))
os.environ["PYTHONPATH"] = str(Path.cwd())
os.environ["PYTHONUNBUFFERED"] = "1"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pandas>=2.3.0"])

import torch
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Ready:", Path.cwd())

## 2. Options

In [ ]:
from pathlib import Path

RUN_FAST = False  # True = smoke test only
RUN_ALL = True      # full finetune (unfrozen)
RUN_FINAL = True    # frozen backbone
RUN_SCRATCH = False # supervised-only baseline

UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

COHORT = "mimic_cohort_aplasia_45_days"
CKPT = Path(
    "MIMIC_IV/saved_data/results/mimic_all/time_series/pretrain/primenet/"
    "fig5_pt_mimicall/fold_0/grid_none"
)
print("Upload files into", UPLOAD_DIR)

## 3. Upload files

Use the Files sidebar → `/content/uploads/` **or** run this cell.

Needed:
- `aplasia_saved_data.zip`
- `checkpoint_best.bin`
- `primenet_saved_variables.pkl`

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

print("Select aplasia_saved_data.zip and/or checkpoint files")
uploaded = files.upload()
for name in uploaded:
    src = Path(name)
    dst = UPLOAD_DIR / src.name
    if src.resolve() != dst.resolve():
        shutil.move(str(src), dst)
    print("staged", dst)

## 4. Place data + checkpoint

In [ ]:
import shutil
import zipfile
from pathlib import Path

root = Path.cwd()
assert root.name == "ChemoTreeVsDL", root

# unzip aplasia_saved_data.zip into repo (paths already start with MIMIC_IV/...)
zips = list(UPLOAD_DIR.glob("*aplasia*.zip")) + list(UPLOAD_DIR.glob("aplasia_saved_data.zip"))
if not zips:
    zips = list(UPLOAD_DIR.glob("*.zip"))
if not zips:
    raise FileNotFoundError(f"No zip in {UPLOAD_DIR}. Upload aplasia_saved_data.zip")

zpath = zips[0]
print("Extracting", zpath)
with zipfile.ZipFile(zpath) as zf:
    zf.extractall(root)

# checkpoint
CKPT.mkdir(parents=True, exist_ok=True)
for name in ("checkpoint_best.bin", "primenet_saved_variables.pkl"):
    hits = list(UPLOAD_DIR.rglob(name))
    if not hits:
        raise FileNotFoundError(f"Missing {name} under {UPLOAD_DIR}")
    shutil.copy2(hits[0], CKPT / name)
    print("ckpt:", hits[0], "→", CKPT / name)

checks = [
    Path(f"MIMIC_IV/saved_data/cohorts/{COHORT}.csv.gz"),
    Path(f"MIMIC_IV/saved_data/top_features/mimic_top100_features.pkl"),
    Path(
        f"MIMIC_IV/saved_data/processed_admission_features_for_ts/{COHORT}/"
        f"{COHORT}_admissions_labs_14_days_to_ts.csv.gz"
    ),
    *[Path(f"MIMIC_IV/saved_data/folds/{COHORT}/fold_{i}.pkl") for i in range(5)],
    CKPT / "checkpoint_best.bin",
    CKPT / "primenet_saved_variables.pkl",
]
missing = [str(p) for p in checks if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing:\n  - " + "\n  - ".join(missing))
print("OK — aplasia data + mimic_all checkpoint ready")

## 5. Finetune

In [ ]:
import subprocess
import sys

def run_folds(prefix: str, *, freeze: bool = False, supervised_only: bool = False):
    for fold in range(5):
        cmd = [
            sys.executable, "-u", "-m", "ts_model_training.main",
            "--dataset", "MIMIC_IV",
            "--cohort", COHORT,
            "--fold", str(fold),
            "--model_type", "primenet",
            "--grid", "none",
            "--feature_threshold",
            "--static_threshold", "0",
            "--hid_dim_demo", "64",
            "--config_path", "config/ts_config_params.yaml",
            "--prefix", prefix,
        ]
        if supervised_only:
            cmd.append("--supervised-only")
        else:
            cmd += ["--load_ckpt_path", str(CKPT)]
        if freeze:
            cmd.append("--freeze")
        if RUN_FAST:
            cmd.append("--fast")
        print("+", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)

if RUN_ALL:
    run_folds("fig5_mimicall_all_aplasia", freeze=False)
if RUN_FINAL:
    run_folds("fig5_mimicall_final_aplasia", freeze=True)
if RUN_SCRATCH:
    run_folds("fig5_none_none_aplasia", supervised_only=True)

print("Finetune finished.")

## 6. Summarize + download zip

In [ ]:
import csv
import shutil
import statistics
from pathlib import Path
from google.colab import files
import pandas as pd
from IPython.display import display

ROOT = Path(
    f"MIMIC_IV/saved_data/results/{COHORT}/time_series/finetune/primenet"
)
SCENARIOS = {
    "mimic_all/all": "fig5_mimicall_all_aplasia",
    "mimic_all/final": "fig5_mimicall_final_aplasia",
    "aplasia/all": "fig5_aplasia_all",
    "aplasia/final": "fig5_aplasia_final",
    # "none/none": "fig5_none_none_aplasia",  # only if you ran scratch
}
METRICS = ["auroc", "auprc", "f1", "precision", "recall", "loss"]

def read_test(p: Path) -> dict:
    with open(p) as f:
        rows = list(csv.DictReader(f))
    test = [r for r in rows if str(r.get("split", "")).lower() == "test"]
    if not test:
        raise FileNotFoundError(f"No test row in {p}")
    return test[-1]

rows = []
for label, prefix in SCENARIOS.items():
    for fold in range(5):
        path = ROOT / prefix / f"fold_{fold}" / "grid_none" / "results_final.csv"
        if not path.is_file():
            print("MISSING", path)
            continue
        r = read_test(path)
        rows.append(
            {
                "scenario": label,
                "fold": fold,
                **{m: float(r[m]) for m in METRICS if m in r and r[m] != ""},
            }
        )

df = pd.DataFrame(rows)
print("=== Per-fold ===")
display(df.round(4) if not df.empty else df)

print("\n=== Mean ± std ===")
if not df.empty:
    display(df.groupby("scenario")[METRICS].agg(["mean", "std", "count"]).round(4))

print("\n=== Paper-style (aplasia) ===")
for scen, g in df.groupby("scenario"):
    parts = []
    for m in ["auroc", "auprc", "f1"]:
        if m in g and len(g) >= 2:
            parts.append(f"{m.upper()} {g[m].mean():.3f}±{g[m].std(ddof=1):.3f}")
        elif m in g and len(g) == 1:
            parts.append(f"{m.upper()} {g[m].iloc[0]:.3f}")
    print(f"{scen:20s}  " + "  ".join(parts) + f"  (n={len(g)})")

staging = Path("/content/aplasia_results_staging")
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()
if ROOT.is_dir():
    shutil.copytree(ROOT, staging / "primenet", dirs_exist_ok=True)
pt = Path(
    f"MIMIC_IV/saved_data/results/{COHORT}/time_series/pretrain/primenet/fig5_pt_aplasia"
)
if pt.is_dir():
    shutil.copytree(pt, staging / "pretrain_fig5_pt_aplasia", dirs_exist_ok=True)

archive = shutil.make_archive("/content/aplasia_finetune_results", "zip", root_dir=staging)
print("\nCreated", archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)")
files.download(archive)
